In [10]:
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
import os
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

### Modifiying JSON

# 1. Normalize Bounding Boxes, One-Hot category encoding

Values bet 0 - 1 (divide by image width/height)

In [3]:
def build_node_features(graph):
    image_W = graph['width']
    image_H = graph['height']

    nodes = graph['nodes']
    features = []

    for node in nodes:
        x, y, w, h = node['bbox']

        #x_norm = x / image_W
        #y_norm = y / image_H
        x_center = (x + w/2) / image_W
        y_center = (y + h/2) / image_H
        w_norm = w / image_W
        h_norm = h / image_H
        area = (w * h) / (image_W * image_H)
        aspect = w / (h + 1e-6)

        category_id = np.zeros(11)
        category_id[node['category_id'] - 1] = 1

        geom_features = np.array([
            x_center,
            y_center,
            w_norm,
            h_norm,
            area,
            aspect
        ])
        
        feature = np.concatenate([geom_features, category_id])

        features.append(feature)

    return np.array(features)

In [34]:
with open("data/train_data/graph_000020.json") as f:
    graph = json.load(f)

features = build_node_features(graph)

print(features.shape)
print(features[0])

(7, 17)
[7.92843930e-01 9.82037881e-01 3.00658480e-01 8.53787121e-03
 2.56698338e-03 3.52146851e+01 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 1.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00]


# 2. Edge Index

In [4]:
def build_edge_index(graph):
    nodes = graph['nodes']
    edges = graph['edges']

    id_map = {}
    for idx, node in enumerate(nodes):
        id_map[node['node_id']] = idx

    edge_list = []
    edge_labels = []

    for edge in edges:
        src = id_map[edge['from']]
        dst = id_map[edge['to']]

        edge_list.append([src, dst])
        edge_labels.append(edge["label"])

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    edge_label = torch.tensor(edge_labels, dtype=torch.float)

    return edge_index, edge_label
    

In [48]:
edge_index, edge_label = build_edge_index(graph)
print(edge_index.shape)
print(edge_index)
print(edge_label.shape)
print(edge_label)

torch.Size([2, 8])
tensor([[1, 1, 1, 1, 2, 2, 2, 2],
        [3, 4, 5, 6, 3, 4, 5, 6]])
torch.Size([8])
tensor([0., 1., 0., 0., 1., 0., 0., 0.])


# 3. kNN graph construction

In [5]:
def build_knn_edges(features, k=3):
    """
    features: numpy array [num_nodes, feature_dim]
    prvé 2 dimenzie musia byť x_center, y_center
    """
    num_nodes = features.shape[0]
    centers = features[:, :2]  # x_center, y_center

    edge_list = []

    for i in range(num_nodes):
        distances = np.linalg.norm(centers - centers[i], axis=1)

        nearest = np.argsort(distances)[1:k+1]  # preskoč sám seba

        for j in nearest:
            edge_list.append([i, j])

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    return edge_index

In [49]:
edge_label_index, edge_label = build_edge_index(graph)
knn_edges = build_knn_edges(features, k=3)

edge_index = torch.cat([edge_label_index, knn_edges], dim=1)

# Load data

In [7]:
def load_all_data(json_folder):
    all_graphs = []

    for filename in os.listdir(json_folder):
        if filename.endswith(".json"):
            with open(os.path.join(json_folder, filename)) as f:
                graph = json.load(f)

            # features
            x = torch.tensor(build_node_features(graph), dtype=torch.float)

            # edge propagation
            knn_edges = build_knn_edges(x.numpy(), k = 5)

            # edge for truth
            true_edge_index, true_edge_labels = build_edge_index(graph)

            # target for node
            node_labels = torch.tensor([1 if n['category_id'] == 9 else 0 for n in graph['nodes']], dtype=torch.long)

            data = Data(
                x=x, 
                edge_index=knn_edges,           # Cez toto tečie GAT
                y_nodes=node_labels,            # Pravda pre uzly
                edge_index_targets=true_edge_index, # Kde majú byť hrany
                y_edges=true_edge_labels        # Pravda pre hrany
            )
            all_graphs.append(data)


    # Pridaj si toto do load_all_data pred return
    #no_edge_count = sum(1 for d in all_graphs if d.edge_index_targets.numel() == 0)
    #print(f"Súborov bez logických väzieb: {no_edge_count} z {len(all_graphs)}")
            

    return all_graphs

# GAT Model

In [14]:
class GATMultitask(nn.Module):
    def __init__(self, in_channels, hidden_channels, heads=4):
        super().__init__()
        # Vrstva 1: Šírenie informácie medzi susedmi (vstup: 17 feat, výstup: hidden * heads)
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        
        # Vrstva 2: Finálna reprezentácia uzlov (vstup: hidden * heads, výstup: hidden)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, concat=False)

        # Klasifikátor pre uzly (Tabuľka vs Ostatné)
        self.node_head = nn.Linear(hidden_channels, 2) 

    def forward(self, x, edge_index):
        # 1. Grafová konvolúcia
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        
        x = self.conv2(x, edge_index)
        
        # Uzly: vrátime logity pre klasifikáciu
        node_logits = self.node_head(x)
        
        return x, node_logits

    def predict_edges(self, z, edge_index):
        # Hrany: skalárny súčin medzi zdrojovým a cieľovým uzlom
        # z[edge_index[0]] sú vektory "odkiaľ", z[edge_index[1]] sú vektory "kam"
        return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=-1)

# Training

## Load all data

In [12]:
# Load graphs
train_graphs = load_all_data("data/train_data")
val_graphs = load_all_data("data/val_data")
test_graphs = load_all_data("data/test_data")

# Create DataLoader
train_loader = DataLoader(train_graphs, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_graphs, batch_size=8, shuffle=False)
test_loader  = DataLoader(test_graphs, batch_size=1, shuffle=False)

print(f"Data are ready: Train({len(train_graphs)}), Val({len(val_graphs)}), Test({len(test_graphs)})")

Data are ready: Train(69103), Val(6480), Test(4994)


In [25]:
# Inicializácia (in_channels = 17, pretože 6 geom + 11 category)
model = GATMultitask(in_channels=17, hidden_channels=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

node_criterion = nn.CrossEntropyLoss()
edge_criterion = nn.BCEWithLogitsLoss()

def train():
    model.train()
    total_loss = 0
    processed_batches = 0

    for batch in train_loader:
        # KONTROLA: Ak batch nemá žiadne cieľové hrany, preskoč výpočet edge loss
        if batch.edge_index_targets.numel() == 0:
            #print(f"Pozor, prázdne hrany v bachi! Tvar: {batch.edge_index_targets.shape}")
            # Ak chceš, môžeš stále trénovať aspoň na uzloch:
            optimizer.zero_grad()
            z, node_logits = model(batch.x, batch.edge_index)
            loss = node_criterion(node_logits, batch.y_nodes)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            processed_batches += 1
            continue

        optimizer.zero_grad()
        
        # 1. Forward pass
        z, node_logits = model(batch.x, batch.edge_index)
        
        # 2. Loss pre uzly
        loss_node = node_criterion(node_logits, batch.y_nodes)
        
        # 3. Loss pre hrany (teraz už vieme, že tam nejaké sú)
        edge_preds = model.predict_edges(z, batch.edge_index_targets)
        loss_edge = edge_criterion(edge_preds, batch.y_edges)
        
        # Spojenie strát
        loss = loss_node + loss_edge
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        processed_batches += 1

    return total_loss / processed_batches

def evaluate_loss(loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            z, node_logits = model(batch.x, batch.edge_index)
            loss_node = node_criterion(node_logits, batch.y_nodes)
            
            # Ošetrenie prázdnych hrán aj vo validácii
            if batch.edge_index_targets.numel() > 0:
                edge_preds = model.predict_edges(z, batch.edge_index_targets)
                loss_edge = edge_criterion(edge_preds, batch.y_edges)
                total_loss += (loss_node + loss_edge).item()
            else:
                total_loss += loss_node.item()
    return total_loss / len(loader)

In [26]:
# Spustenie tréningu
for epoch in range(1, 31): # Skúsme 30 epoch
    train_loss = train() # Použijeme tú upravenú train() funkciu z predošlej správy
    val_loss = evaluate_loss(val_loader)
    
    print(f'Epoch: {epoch:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

Epoch: 001 | Train Loss: 0.5294 | Val Loss: 0.2912
Epoch: 002 | Train Loss: 0.4383 | Val Loss: 0.2595
Epoch: 003 | Train Loss: 0.4113 | Val Loss: 0.2502
Epoch: 004 | Train Loss: 0.3963 | Val Loss: 0.2402
Epoch: 005 | Train Loss: 0.3818 | Val Loss: 0.2244
Epoch: 006 | Train Loss: 0.3775 | Val Loss: 0.2170
Epoch: 007 | Train Loss: 0.3755 | Val Loss: 0.2404
Epoch: 008 | Train Loss: 0.3641 | Val Loss: 0.2693
Epoch: 009 | Train Loss: 0.3707 | Val Loss: 0.2050
Epoch: 010 | Train Loss: 0.3592 | Val Loss: 0.2000
Epoch: 011 | Train Loss: 0.3619 | Val Loss: 0.1939
Epoch: 012 | Train Loss: 0.3581 | Val Loss: 0.2320
Epoch: 013 | Train Loss: 0.3585 | Val Loss: 0.2792
Epoch: 014 | Train Loss: 0.3516 | Val Loss: 0.2286
Epoch: 015 | Train Loss: 0.3542 | Val Loss: 0.1968
Epoch: 016 | Train Loss: 0.3464 | Val Loss: 0.3232
Epoch: 017 | Train Loss: 0.3486 | Val Loss: 0.2178
Epoch: 018 | Train Loss: 0.3508 | Val Loss: 0.2015
Epoch: 019 | Train Loss: 0.3532 | Val Loss: 0.2088
Epoch: 020 | Train Loss: 0.3475

# Evaluation

In [29]:
from sklearn.metrics import classification_report, confusion_matrix

def final_evaluation(loader):
    model.eval()
    
    # Pre uzly
    all_node_preds = []
    all_node_labels = []
    
    # Pre hrany
    all_edge_preds = []
    all_edge_labels = []
    
    with torch.no_grad():
        for batch in loader:

            if batch.edge_index_targets.numel() == 0:
                continue
            # 1. Získaj embeddingy a predikcie uzlov
            z, node_logits = model(batch.x, batch.edge_index)
            
            # Uzly - klasifikácia
            node_preds = node_logits.argmax(dim=1)
            all_node_preds.extend(node_preds.cpu().numpy())
            all_node_labels.extend(batch.y_nodes.cpu().numpy())
            
            # 2. Hrany - vyhodnotenie pravdivých cieľov (targets)
            if batch.edge_index_targets.numel() > 0:
                # Získaj skóre pre hrany (sigmoid premení logity na pravdepodobnosť 0-1)
                edge_scores = torch.sigmoid(model.predict_edges(z, batch.edge_index_targets))
                edge_preds = (edge_scores > 0.5).float() # Prah 0.5 pre existenciu hrany
                
                all_edge_preds.extend(edge_preds.cpu().numpy())
                all_edge_labels.extend(batch.y_edges.cpu().numpy())
    
    print("\n--- VÝSLEDKY PRE UZLY (Tabuľka vs Ostatné) ---")
    print(classification_report(all_node_labels, all_node_preds, target_names=['Ostatné', 'Tabuľka']))
    
    if len(all_edge_labels) > 0:
        print("\n--- VÝSLEDKY PRE HRANY (Logické prepojenia) ---")
        # Tu predpokladáme, že y_edges sú 0 a 1
        print(classification_report(all_edge_labels, all_edge_preds))
    else:
        print("\n--- VÝSLEDKY PRE HRANY: Žiadne cieľové hrany v testovacom sete ---")

# Spusti znova
final_evaluation(test_loader)


--- VÝSLEDKY PRE UZLY (Tabuľka vs Ostatné) ---
              precision    recall  f1-score   support

     Ostatné       1.00      0.99      0.99     10824
     Tabuľka       0.75      0.94      0.84       347

    accuracy                           0.99     11171
   macro avg       0.88      0.96      0.91     11171
weighted avg       0.99      0.99      0.99     11171


--- VÝSLEDKY PRE HRANY (Logické prepojenia) ---
              precision    recall  f1-score   support

         0.0       0.89      0.73      0.80      3060
         1.0       0.60      0.82      0.69      1479

    accuracy                           0.76      4539
   macro avg       0.74      0.77      0.75      4539
weighted avg       0.80      0.76      0.77      4539



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_prediction(data_obj):
    model.eval()
    with torch.no_grad():
        # Predikcia
        _, node_logits = model(data_obj.x, data_obj.edge_index)
        preds = node_logits.argmax(dim=1).numpy()
    
    # x obsahuje: [x_center, y_center, w_norm, h_norm, area, aspect, ...cat_id...]
    features = data_obj.x.numpy()
    
    fig, ax = plt.subplots(1, figsize=(8, 10))
    ax.set_title("Predikcia: Červená = Tabuľka, Zelená = Iné")

    for i in range(len(preds)):
        # Rekonštrukcia bboxu z normalizovaných súradníc (zjednodušene pre vizualizáciu)
        xc, yc, w, h = features[i, 0], features[i, 1], features[i, 2], features[i, 3]
        
        # Prepočet na rohy pre matplotlib
        rect = patches.Rectangle(
            (xc - w/2, yc - h/2), w, h, 
            linewidth=2, 
            edgecolor='red' if preds[i] == 1 else 'green', 
            facecolor='none', 
            alpha=0.6
        )
        ax.add_patch(rect)

    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0) # Prevrátená os Y (ako v obrázkoch)
    plt.show()

# Otestuj na prvom grafe z testovacej sady
visualize_prediction(test_data[0])